<a href="https://colab.research.google.com/github/StathisDevves/Industrial/blob/main/Industrial_MiniMill_7Days_March2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# =========================================================
# MINI-MILL 7-DAY OPTIMIZATION
# =========================================================
# INPUT FILE:
#   Electricity_Prices_2025_filtered_8760.xlsx
#
# REQUIRED TIME WINDOW:
#   from 2025-03-19 00:00
#   to   2025-03-25 23:00
#
# LOGIC:
#   - Select hourly prices for the 7-day period
#   - For each day independently:
#       choose the start hour of EAF1.1 that minimizes total 24-hour cost
#       over the fixed 24-process sequence
#   - Create a global process-time index t from 1 to 168
#   - Produce an Excel workbook with:
#       Summary
#       Daily Optimization
#       Optimized Schedule
#       Selected Prices
#       Notes
#
# IMPORTANT:
#   This code assumes the source file has a datetime column and a price column.
#   The parser below tries to detect them automatically.
# =========================================================

input_file = "Electricity_Prices_2025_filtered_8760.xlsx"
output_file = "Mini_Mill_7Day_Optimized_Schedule_2025-03-19_to_2025-03-25.xlsx"

start_ts = pd.Timestamp("2025-03-19 00:00")
end_ts   = pd.Timestamp("2025-03-25 23:00")

# ---------------------------------------------------------
# 1. READ INPUT
# ---------------------------------------------------------
raw = pd.read_excel(input_file)

if raw.empty:
    raise ValueError("Input file is empty.")

raw.columns = [str(c).strip() for c in raw.columns]

# ---------------------------------------------------------
# 2. DETECT DATETIME COLUMN
# ---------------------------------------------------------
datetime_col = None
for col in raw.columns:
    col_lower = col.lower()
    if any(key in col_lower for key in ["datetime", "date", "time", "timestamp"]):
        try:
            test = pd.to_datetime(raw[col], errors="coerce")
            if test.notna().sum() > 0:
                datetime_col = col
                raw[col] = test
                break
        except Exception:
            pass

if datetime_col is None:
    # Fallback: try first column
    test = pd.to_datetime(raw.iloc[:, 0], errors="coerce")
    if test.notna().sum() > 0:
        datetime_col = raw.columns[0]
        raw[datetime_col] = test
    else:
        raise ValueError("Could not detect a valid datetime column.")

# ---------------------------------------------------------
# 3. DETECT PRICE COLUMN
# ---------------------------------------------------------
price_col = None
for col in raw.columns:
    if col == datetime_col:
        continue
    col_lower = col.lower()
    if "price" in col_lower:
        price_col = col
        break

if price_col is None:
    # fallback: first numeric column after datetime
    for col in raw.columns:
        if col == datetime_col:
            continue
        if pd.api.types.is_numeric_dtype(raw[col]):
            price_col = col
            break

if price_col is None:
    raise ValueError("Could not detect a valid price column.")

df = raw[[datetime_col, price_col]].copy()
df.columns = ["Datetime", "Price"]
df = df.dropna(subset=["Datetime", "Price"]).copy()
df["Datetime"] = pd.to_datetime(df["Datetime"])
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df = df.dropna(subset=["Price"]).copy()
df = df.sort_values("Datetime").reset_index(drop=True)

# ---------------------------------------------------------
# 4. FILTER REQUIRED 7-DAY WINDOW
# ---------------------------------------------------------
selected = df[(df["Datetime"] >= start_ts) & (df["Datetime"] <= end_ts)].copy()

expected_hours = 24 * 7
if len(selected) != expected_hours:
    raise ValueError(
        f"Expected {expected_hours} hourly rows in the selected period, found {len(selected)}."
    )

selected["Date"] = selected["Datetime"].dt.date
selected["Hour"] = selected["Datetime"].dt.hour

# ---------------------------------------------------------
# 5. DEFINE FIXED 24-PROCESS SEQUENCE
# ---------------------------------------------------------
stage_sequence = [
    ("Step 1", "EAF1.1", 72, 1),
    ("Step 1", "EAF1.2", 78, 2),
    ("Step 1", "EAF1.3", 80, 3),
    ("Step 1", "EAF1.4", 72, 4),

    ("Step 2", "Secondary Downstream 1.1", 32, 5),
    ("Step 2", "Secondary Downstream 1.2", 26, 6),
    ("Step 2", "Secondary Downstream 1.3", 20, 7),

    ("Step 3", "SD sequence Blue Colour", 15, 8),

    ("Step 4", "Minimum Critical Load 1", 12, 9),
    ("Step 4", "Minimum Critical Load 2", 12, 10),
    ("Step 4", "Minimum Critical Load 3", 12, 11),
    ("Step 4", "Minimum Critical Load 4", 12, 12),
    ("Step 4", "Minimum Critical Load 5", 12, 13),
    ("Step 4", "Minimum Critical Load 6", 12, 14),

    ("Step 5", "Preparation Blue 1", 15, 15),
    ("Step 5", "Preparation Blue 2", 24, 16),
    ("Step 5", "Preparation Blue 3", 38, 17),

    ("Step 6", "EAF2.1", 72, 18),
    ("Step 6", "EAF2.2", 78, 19),
    ("Step 6", "EAF2.3", 80, 20),
    ("Step 6", "EAF2.4", 76, 21),

    ("Step 7", "Secondary Downstream 2.1", 28, 22),
    ("Step 7", "Secondary Downstream 2.2", 24, 23),
    ("Step 7", "Secondary Downstream 2.3", 20, 24),
]

df_sequence = pd.DataFrame(
    stage_sequence,
    columns=["Step", "Production Phase", "Total Load (MWh)", "t_in_day"]
)

# ---------------------------------------------------------
# 6. BUILD ONE-DAY SCHEDULE
# ---------------------------------------------------------
def build_day_schedule(day_df, start_hour):
    """
    day_df: one-day DataFrame with 24 hours
    start_hour: hour index 0..23 for EAF1.1
    """
    prices = day_df["Price"].tolist()
    datetimes = day_df["Datetime"].tolist()
    hours = day_df["Hour"].tolist()

    rows = []
    total_cost = 0.0

    for i, (step, phase, load, t_in_day) in enumerate(stage_sequence):
        idx = (start_hour + i) % 24   # circular within each day
        dt = datetimes[idx]
        hr = hours[idx]
        price = prices[idx]
        cost = load * price
        total_cost += cost

        rows.append({
            "t_in_day": t_in_day,
            "Step": step,
            "Production Phase": phase,
            "Total Load (MWh)": load,
            "Assigned Hour": hr,
            "Assigned Datetime": dt,
            "Price": price,
            "Cost = Load x Price": cost
        })

    return pd.DataFrame(rows), total_cost

# ---------------------------------------------------------
# 7. OPTIMIZE EACH DAY
# ---------------------------------------------------------
daily_optimization_rows = []
optimized_schedule_rows = []

dates = sorted(selected["Date"].unique())
global_t = 1
total_7day_cost = 0.0

for d in dates:
    day_df = selected[selected["Date"] == d].sort_values("Hour").reset_index(drop=True)

    if len(day_df) != 24:
        raise ValueError(f"Date {d} does not contain 24 hourly rows.")

    best_start = None
    best_cost = None
    best_schedule = None

    for start_hour in range(24):
        schedule_df, total_cost = build_day_schedule(day_df, start_hour)

        if best_cost is None or total_cost < best_cost:
            best_cost = total_cost
            best_start = start_hour
            best_schedule = schedule_df.copy()

    total_7day_cost += best_cost

    daily_optimization_rows.append({
        "Date": pd.Timestamp(d),
        "Optimal Start Hour (EAF1.1)": best_start,
        "Minimum Daily Cost": best_cost
    })

    for _, row in best_schedule.iterrows():
        optimized_schedule_rows.append({
            "Global t": global_t,
            "Date": pd.Timestamp(d),
            "t_in_day": int(row["t_in_day"]),
            "Step": row["Step"],
            "Production Phase": row["Production Phase"],
            "Total Load (MWh)": row["Total Load (MWh)"],
            "Assigned Hour": row["Assigned Hour"],
            "Assigned Datetime": row["Assigned Datetime"],
            "Price": row["Price"],
            "Cost = Load x Price": row["Cost = Load x Price"]
        })
        global_t += 1

df_daily_optimization = pd.DataFrame(daily_optimization_rows)
df_optimized_schedule = pd.DataFrame(optimized_schedule_rows)

# ---------------------------------------------------------
# 8. SUMMARY + NOTES
# ---------------------------------------------------------
df_summary = pd.DataFrame({
    "Metric": [
        "Selected start datetime",
        "Selected end datetime",
        "Number of days optimized",
        "Total process times",
        "7-day total optimized cost (EUR)"
    ],
    "Value": [
        start_ts,
        end_ts,
        7,
        24 * 7,
        total_7day_cost
    ]
})

df_notes = pd.DataFrame({
    "Notes": [
        "The model selects hourly prices from 2025-03-19 00:00 to 2025-03-25 23:00.",
        "Each day is optimized independently as one 24-process mini-mill circle.",
        "Within each day, the sequence is mapped circularly over the 24 hourly prices.",
        "Global process time runs from t=1 to t=168.",
        "The objective minimizes the sum of Load x Price for each day separately, then aggregates the 7-day total."
    ]
})

# ---------------------------------------------------------
# 9. WRITE EXCEL
# ---------------------------------------------------------
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="Summary", index=False)
    df_daily_optimization.to_excel(writer, sheet_name="Daily Optimization", index=False)
    df_optimized_schedule.to_excel(writer, sheet_name="Optimized Schedule", index=False)
    selected[["Datetime", "Date", "Hour", "Price"]].to_excel(writer, sheet_name="Selected Prices", index=False)
    df_notes.to_excel(writer, sheet_name="Notes", index=False)

# ---------------------------------------------------------
# 10. FORMAT EXCEL
# ---------------------------------------------------------
wb = load_workbook(output_file)

header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
thin = Side(style="thin", color="BFBFBF")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

phase_fills = {
    "EAF": PatternFill("solid", fgColor="FCE4D6"),
    "Secondary": PatternFill("solid", fgColor="E2F0D9"),
    "SD sequence Blue Colour": PatternFill("solid", fgColor="D9EAF7"),
    "Minimum Critical Load": PatternFill("solid", fgColor="F4CCCC"),
    "Preparation Blue": PatternFill("solid", fgColor="D9E1F2"),
}

for ws in wb.worksheets:
    # Header style
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border

    # Body style
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.border = border
            cell.alignment = Alignment(vertical="center")

    # Auto-width
    for col_cells in ws.columns:
        max_len = 0
        col_letter = col_cells[0].column_letter
        for cell in col_cells:
            val = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 35)

# Number formatting
for sheet_name in ["Daily Optimization", "Optimized Schedule", "Selected Prices", "Summary"]:
    ws = wb[sheet_name]
    headers = [c.value for c in ws[1]]
    for col_idx, header in enumerate(headers, start=1):
        if header and ("Price" in str(header) or "Cost" in str(header)):
            for row in range(2, ws.max_row + 1):
                ws.cell(row=row, column=col_idx).number_format = "0.00"

# Datetime formatting
for sheet_name in ["Optimized Schedule", "Selected Prices", "Summary", "Daily Optimization"]:
    ws = wb[sheet_name]
    headers = [c.value for c in ws[1]]
    for col_idx, header in enumerate(headers, start=1):
        if header and ("Datetime" in str(header) or "Date" in str(header)):
            for row in range(2, ws.max_row + 1):
                val = ws.cell(row=row, column=col_idx).value
                if val is not None:
                    try:
                        ws.cell(row=row, column=col_idx).number_format = "yyyy-mm-dd hh:mm"
                    except Exception:
                        pass

# Color production phases
ws_opt = wb["Optimized Schedule"]
headers_opt = [c.value for c in ws_opt[1]]
phase_col = headers_opt.index("Production Phase") + 1

for row in range(2, ws_opt.max_row + 1):
    phase_value = str(ws_opt.cell(row=row, column=phase_col).value)

    fill_to_use = None
    if phase_value.startswith("EAF"):
        fill_to_use = phase_fills["EAF"]
    elif phase_value.startswith("Secondary"):
        fill_to_use = phase_fills["Secondary"]
    elif phase_value == "SD sequence Blue Colour":
        fill_to_use = phase_fills["SD sequence Blue Colour"]
    elif phase_value.startswith("Minimum Critical Load"):
        fill_to_use = phase_fills["Minimum Critical Load"]
    elif phase_value.startswith("Preparation Blue"):
        fill_to_use = phase_fills["Preparation Blue"]

    if fill_to_use:
        ws_opt.cell(row=row, column=phase_col).fill = fill_to_use

wb.save(output_file)

# ---------------------------------------------------------
# 11. PRINT RESULTS
# ---------------------------------------------------------
print(f"Output saved to: {output_file}")
print(f"7-day total optimized cost: {total_7day_cost:,.2f} EUR")
print("\nDaily optimal start hours:")
for _, row in df_daily_optimization.iterrows():
    print(f"{row['Date'].date()} -> {int(row['Optimal Start Hour (EAF1.1)'])}")

Output saved to: Mini_Mill_7Day_Optimized_Schedule_2025-03-19_to_2025-03-25.xlsx
7-day total optimized cost: 548,642.56 EUR

Daily optimal start hours:
2025-03-19 -> 8
2025-03-20 -> 8
2025-03-21 -> 8
2025-03-22 -> 10
2025-03-23 -> 11
2025-03-24 -> 8
2025-03-25 -> 8
